# Get to Know the elDORS (version-1) database

This notebook serves as a guided tour of the [elDORS (v1) database](https://registry.opendata.aws/eldors-v1). More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

### Organization for the database distributions (and pipeline builds): the key prefix structure of the cooreponding S3 bucket.

At the top level of our S3 bucket, the 3.2 TB dataset is divided into four primary prefixes (distributions) to suit different computational environments:

 1. `elDORS_v1_raw/`: Raw (unclustered) version of the database provided in ~9GB sequence-aware chunks (.fasta.gz format).
 2. `elDORS_v1/`: 80% sequence-identity (considerimg 80% overlap) clustered version provided in ~9GB sequence-aware chunks (.fasta.gz format).
 3. `rMSA_optimized_elDORS/`: Optimized database build (BLAST and unzipped FASTA formats) suitable for the rMSA pipeline.
 4. `RNAcmap3_optimized_elDORS/`: Optimized split-strategy database (BLAST and unzipped FASTA formats) suitable for the RNAcmap3 pipeline.
 
 Full documentation for this dataset can be found at: (https://github.com/duttan710/elDORS-aws-opendata/blob/main/elDORS_Documentation.md)

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# biopython >= 1.81
# matplotlib >= 3.10.3

# Import the libraries required for this notebook
import os
import boto3
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config
from Bio import SeqIO

In [ ]:
# First, we will define the location of our dataset, create our boto3 S3 client, and list the top-level prefixes in our S3 bucket to verify the four distributions.
# Location of the S3 bucket for this dataset
bucket = "<PENDING-AWS-BUCKET-NAME>"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
print("Top-level distributions in elDORS:")
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(f"- {item['Prefix']}")

### Data formats present in the dataset
### Q: What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Our dataset primarily consists of **FASTA** format files, along with specialized indexed format provided for downstrem pipelines (**BLAST** format files).

Generally, FASTA is a text-based format for representing either nucleotide sequences or amino acid sequences, in which nucleobases or amino acids are represented using single-letter codes. It begins with a single-line description (starting with a `>`), followed by lines of sequence data.

Our dataset uses this format because:
 - It is the universal standard for bioinformatics and sequence alignment.
 - It easily handles massive biological sequences in a human-readable and machine-parsable format.
 - For the optimizations of the database (elDORS_v1) for efficient use with Multiple Sequence Alignment pipelines, it is also provided in multi-volume BLAST format along with the corresponding multi-volume (uncompressed) FASTA files
 
FASTA files are natively supported by almost all bioinformatics tools (like NCBI-BLAST, NMHHER, INFERNAL, rMSA and RNAcmap3) and can be easily processed in Python using the `Biopython` library.

BLAST format files provided here can be used with MSA generation pipelines rMSA and RNAcmap3 as well as sequence search using NCBI-BLAST

### An example of downloading and loading data from the dataset

As an example, let us download a single chunk from the clustered distribution to explore its contents locally. We will use the AWS CLI via a bash command for efficient downloading.

In [ ]:
%%bash
# Download a sample FASTA chunk (Note: replace 'chunk_001.fasta' with an actual filename from your bucket)
aws s3 cp s3://<PENDING-AWS-BUCKET-NAME>/GZIPPED_elDORS_v1/elDORS_v1_chunks/chunk_001.fasta.gz ./sample_data/ --no-sign-request

# Unzip for local analysis
gunzip ./sample_data/chunk_001.fasta.gz
ls -lh ./sample_data/

### 1. Verifying File Integrity after the .fasta.gz chunks are downloaded in desired location

Before using the data, it is highly recommended to verify that no files were corrupted or truncated during the download or transfer process. Both the raw and clustered databases include a SHA256 manifest file for this purpose.



In [ ]:
%%bash
#For the unclustered version of the database (elDORS_v1_raw):
sha256sum -c elDORS_raw_v1_manifest.sha256

In [ ]:
%%bash
#For the clustered version of the database (elDORS_v1):
sha256sum -c elDORS_v1_manifest.sha256

## 2. Decompression & Assembly

To decompress and merge the chunks into a single uncompressed FASTA file, run the following command:

In [ ]:
%%bash
#For the unclustered version of the database (elDORS_v1_raw):
pigz -dc -p 16 elDORS_v1_raw*.fasta.gz > elDORS_v1_raw_complete.fasta # conda install pigz (if not installed)

In [ ]:
%%bash
#For the clustered version of the database (elDORS_v1):
pigz -dc -p 16 elDORS_v1*.fasta.gz > elDORS_v1_complete.fasta

### Dynamic Inspection of a Downloaded Data Volume/Volumes
Beyond the high-level architecture, users can dynamically parse individual volumes using Python. Below, we plot the sequence length distribution of the sample FASTA chunk we downloaded in the previous step.

In [ ]:
# Parse the FASTA file and extract sequence lengths
fasta_file = "./sample_data/chunk_001.fasta"
sequence_lengths = [len(record.seq) for record in SeqIO.parse(fasta_file, "fasta")]

# Plot using matplotlib
plt.figure(figsize=(10, 5), dpi=100, facecolor='white')

plt.hist(sequence_lengths, 
         bins=50,
         color='#2ecc71',
         edgecolor='white',
         linewidth=1.2,
         alpha=0.8)

plt.title('Distribution of RNA Sequence Lengths in Sample Chunk', 
         fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Sequence Length (Nucleotides)', fontsize=11, labelpad=10)
plt.ylabel('Count', fontsize=11, labelpad=10)

plt.grid(True, linestyle='--', alpha=0.3, color='gray')
ax = plt.gca()
ax.set_facecolor('#f8f9fa')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Total sequences analyzed in this sample volume: {len(sequence_lengths)}")

### Application of the database to downstream bioinformatics pipelines?

**Answer:** The elDORS database is explicitly engineered to fuel deep RNA Multiple Sequence Alignment (MSA). Our optimized builds act as drop-in replacements for legacy databases in `rMSA` and `RNAcmap3` pipelines. These deep MSAs are then utilized for Direct Coupling Analysis (DCA), 2D/3D structure prediction, and RNA language model (LM) training.

* Please refer to the documention for [diagramatic overview](https://github.com/duttan710/elDORS-aws-opendata/blob/main/elDORS_Documentation.md)

Because institutional or particular regional servers hosting legacy RNA databases are sometimes subject to regional network restrictions or severe latency timeouts, the AWS-hosted elDORS builds ensure researchers globally can execute these pipelines locally without interruption.

### Prerequisites: Tool Installations & Environment Setup for MSA generation pipelines

The elDORS dataset provides the massive, optimized sequence infrastructure required for deep RNA MSA generation. However, users are responsible for independently installing, configuring, and activating the downstream pipelines in their local environments (e.g., via Conda). 

Please refer to the official repositories for installation instructions (also for the installation of the additional tools/dendencies required by the respective pipelines), and ensure you cite the original authors when utilizing their tools:

*   **RNAcmap3-elDORS Pipeline:** The RNAcmap3 pipeline relies on the [RNAcmap2 search architecture](https://github.com/jaswindersingh2/RNAcmap2) but traditionally utilizes the MARS database. By replacing MARS with our optimized builds, you are executing the **RNAcmap3-elDORS** pipeline benchmarked in our manuscript (https://www.biorxiv.org/content/10.64898/2026.07.10.737016v1).
 
*   **rMSA:** [pylelab/rMSA] (https://github.com/pylelab/rMSA)

* Primary citations:

Singh, J., Paliwal, K., Singh, J., Litfin, T., and Zhou, Y. (2022). Improved RNA homology detection and alignment by automatic iterative search in an expanded database. bioRxiv 2022.10.03.510702; doi: https://doi.org/10.1101/2022.10.03.510702.

Chen, K., Litfin, T., Singh, J., Zhan, J., & Zhou, Y. (2024). MARS and RNAcmap3: The Master Database of All Possible RNA Sequences Integrated with RNAcmap for RNA Homology Search. Genomics, proteomics & bioinformatics, 22(1), qzae018. https://doi.org/10.1093/gpbjnl/qzae018.

Zhang, C., Zhang, Y., & Pyle, A. M. (2022). rMSA: A sequence search and alignment algorithm to improve RNA structure modeling. Journal of Molecular Biology, 435(49), Article 167904. https://doi.org/10.1016/j.jmb.2022.167904.



### Integrating elDORS into MSA generation Pipelines

To use the optimized elDORS builds with your existing pipelines, you do not need to reinstall the underlying tools. You simply need to point the pipeline's database variable to your downloaded AWS S3 directory.

#### For rMSA Users: Modifying the Wrapper Script
If you are using the standard `rMSA` wrapper scripts, open the main execution script (e.g., `rMSA.pl` or the equivalent configuration file) in a text editor. 

Locate the database directory variables at the top of the script. You will keep your standard Rfam and RNAcentral paths, but you will update the main sequence database variable (often `$db2` or `$dbnewdir`) to point to your downloaded `rMSA_optimized_elDORS` path.

**Updated Config for elDORS (`rMSA.pl`):**
```perl
#!/usr/bin/perl
use strict;
use File::Basename;
use Cwd 'abs_path';

my $rootdir=dirname(abs_path(__FILE__));
my $bindir ="$rootdir/bin";
my $dbdir  ="$rootdir/database";

# Point to your downloaded AWS elDORS directory
my $dbnewdir ="/absolute/path/to/rMSA_optimized_elDORS"; 

my $db0    ="$dbdir/Rfam.cm";
my $db1    ="$dbdir/rnacentral.fasta";

# Target the elDORS v1 database prefix
my $db2    ="$dbnewdir/elDORS_v1_db"; 
```

#### For RNAcmap3 Users: Creating a Run Script
RNAcmap3 utilizes a split-strategy that requires both a BLAST-formatted database and an Infernal-compatible sequence database. The `RNAcmap3_optimized_elDORS` distribution is pre-packaged with both. 

You can use the following generalized bash template to execute RNAcmap3 by pointing the `-b` (BLAST) and `-c` (Infernal) flags to your downloaded directories.

**Generalized RNAcmap3-elDORS Execution Template:**
```bash
#!/bin/bash
# Generalized template for running RNAcmap3 with elDORS v1

# 0. Activate the rna_cmap2 environment
# conda activate rna_cmap2

# 1. Set your input sequence and computational resources
INPUT_FASTA="your_target_sequence.fasta"
THREADS=8 #change the number of threads as per requirement

# 2. Point to your downloaded AWS elDORS directories
# NOTE: Update "/absolute/path/to/" to match your actual download location
ELDORS_DIR="/absolute/path/to/RNAcmap3_optimized_elDORS"

# The BLAST database prefix inside the blast/ directory
BLAST_DB="${ELDORS_DIR}/blast/elDORS_v1_db"

# The Infernal directory containing the uncompressed FASTA volumes
INFERNAL_DB="${ELDORS_DIR}/infernal"

# 3. Execute the pipeline
# -i: Input FASTA
# -n: Number of threads
# -b: BLAST database path
# -c: Infernal database path
# Add additional flags (e.g., -d gremlin) as required by your specific workflow
bash /path/to/your/RNAcamp3/run_rnacmap.sh \
    -i "$INPUT_FASTA" \
    -n "$THREADS" \
    -b "$BLAST_DB" \
    -c "$INFERNAL_DB"


### 3. Other homologous sequence search pipelines

Once you have assembled the `elDORS_v1_complete.fasta` file, it can be seamlessly adapted for use with standard MSA generation tools. Below are examples of how to format and query the database using BLAST, NHMMER, and INFERNAL.

* ** Option-1 (Recommended - pre-indexed):** Skip the heavy computational indexing step entirely! If you are planning to use BLAST format database, you can directly download the pre-compiled, multi-volume optimizations of the database from our S3 bucket (`rMSA_optimized_elDORS/` or `RNAcmap3_optimized_elDORS/`).

* ** Option-2 (Local Adaptation):** If you prefer to build a custom local adaptation of the database, you can build the database indexes manually using your locally assembled `elDORS_v1_complete.fasta` file.

### Prerequisites: Installation of the tools and dependencies

In [ ]:
%%bash
# You can install BLAST+, HMMER, and INFERNAL using conda
conda install -c conda-forge -c bioconda blast hmmer infernal


Please refer to the official repositories for further installation instructions (also for the installation of the additional tools/dendencies required by the respective pipelines), and ensure you cite the original authors when utilizing their tools:

Primary Citations:

Altschul, S.F., Madden, T.L., Schäffer, A.A., Zhang, J., Zhang, Z., Miller, W. and Lipman, D.J. (1997). Gapped BLAST and PSI-BLAST: a new generation of protein database search programs. Nucleic acids research, 25(17), pp.3389-3402.https://doi.org/10.1093/nar/25.17.3389. 

Nawrocki, E.P. and Eddy, S.R., (2013). Infernal 1.1: 100-fold faster RNA homology searches. Bioinformatics, 29(22):2933-2935. https://doi.org/10.1093/bioinformatics/btt509.

Eddy SR.  (2009). A new generation of homology search tools based on probabilistic inference. Genome Inform. 23:205-11. https://doi.org/10.1142/9781848165632_0019.

Wheeler, T.J., Eddy, S.R. (2013). nhmmer: DNA homology search with profile HMMs. Bioinformatics, 29(19):2487–2489, https://doi.org/10.1093/bioinformatics/btt403



#### A. Using BLAST 
To use BLAST, you must first build a BLAST database from the assembled FASTA file (Option-2) or use a pre-indexed build ("RNAcmap3_optimized_elDORS/blast/elDORS_v1/" or "rMSA_optimized_elDORS/elDORS_v1_db")

In [ ]:
%%bash
#1 Create the BLAST Database:

makeblastdb -in elDORS_v1_complete.fasta -dbtype nucl -parse_seqids -out elDORS_v1_blast

#2 Example of running a search (blastn):
blastn -query your_query.fasta -db elDORS_v1_blast -outfmt 6 -out blast_results.tsv -num_threads 16  # "blastn -h" for help

#3 Retrieve Specific Sequences (blastdbcmd): (Extract specific hits from the BLAST format database using their sequence IDs)
blastdbcmd -db elDORS_v1_blast -entry "SPECIFIC_HIT_ID" > extracted_hit.fasta



#### B.  Using NHMMER 
The nhmmer tool (part of the HMMER suite) is optimized for searching nucleotide sequences using Profile Hidden Markov Models (HMMs). You can search a query sequnce (in FASTA format)/HMM profile/existing alignment directly against the uncompressed elDORS_v1_complete.fasta file.

In [ ]:
%%bash
#2 Example of running a search (nhmmer):
nhmmer --tblout nhmmer_results.tbl -A nhmmer_alignment.sto -cpu 16 query.fasta elDORS_v1_complete.fasta # "nhmmer -h" for help

#### B. Using INFERNAL 
INFERNAL allows you to search for RNA consensus sequences and secondary structures simultaneously using Covariance Models (CMs).

In [ ]:
%%bash
#3 Example of running a search (cmsearch):
cmsearch --tblout infernal_results.tbl --cpu 16 your_covariance_model.cm elDORS_v1_complete.fasta # "cmsearch -h" for help